# Retrieval Bias and Epistemic Inequality in Retrieval-Augmented Generation

Reference implementation. This notebook defines the generative model of the RAG pipeline
and reproduces every experiment: the main comparison, the phase maps, the Shapley
attribution, the mitigation analysis, the robustness check, and the retriever-bias
calibration on FLORES-200.

Sections 1-5 are offline and seed-locked. Set `FAST = False` to reproduce the reported
numbers; `FAST = True` runs a coarser grid for quick inspection. Section 6 downloads
FLORES-200 and four sentence embedders and therefore requires internet access.

In [ ]:
import numpy as np, itertools, time
from math import factorial
from functools import lru_cache
from dataclasses import dataclass, replace
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10.5, "axes.titlesize": 11.5,
                     "axes.labelsize": 10.5, "figure.autolayout": True})
CN, CS = "#1f3b73", "#c0392b"
CR, CB, CP = "#4C72B0", "#c0392b", "#55A868"
F32 = np.float32
FAST = False   # keep False to reproduce the reference numbers exactly


from dataclasses import dataclass, replace
import itertools
from math import factorial
import numpy as np

F32 = np.float32


@dataclass(frozen=True)
class Config:
    # latent space / world size
    d: int = 64
    n_topics: int = 200
    frac_south_topics: float = 0.5
    queries_per_topic: int = 10

    # corpus
    docs_per_north_topic: int = 20
    rho: float = 0.05                 # Southern representation ratio
    stochastic_coverage: bool = True  # docs per Southern topic ~ Binomial(n_D, rho)
    south_doc_noise_mult: float = 2.0 # quality degradation of Southern docs
    n_distractors: int = 3000

    # geometry
    gamma: float = 0.55
    query_noise: float = 0.35
    doc_noise: float = 0.25

    # retriever (content-based bias: score += beta * (d . c_N))
    beta: float = 0.30
    k: int = 5
    tau: float = 0.10

    # generator
    alpha: float = 0.70
    pi: float = 0.60                  # Southern prior skew toward c_N
    pi_north: float = 0.0             # Northern prior skew toward c_N
    prior_noise: float = 0.30

    # mitigation
    exposure_floor: int = 0           # floor filled by BIASED score within
                                      # Southern-origin docs (no oracle)
    seed: int = 0


def _unit(x, axis=-1):
    n = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / np.where(n == 0.0, 1.0, n)


@dataclass
class World:
    cfg: Config
    a: np.ndarray
    topic_is_south: np.ndarray
    c_N: np.ndarray
    q: np.ndarray
    q_topic: np.ndarray
    q_is_south: np.ndarray
    doc_vec: np.ndarray
    doc_is_south_info: np.ndarray
    frac_south_topics_uncovered: float


def build_world(cfg: Config) -> World:
    rng = np.random.default_rng(cfg.seed)
    d, M = cfg.d, cfg.n_topics

    c_N = _unit(rng.standard_normal(d)).astype(F32)

    n_south = int(round(cfg.frac_south_topics * M))
    topic_is_south = np.zeros(M, dtype=bool)
    topic_is_south[:n_south] = True
    rng.shuffle(topic_is_south)

    R = rng.standard_normal((M, d)).astype(F32)
    R = _unit(R - (R @ c_N)[:, None] * c_N[None, :])
    sign = np.where(topic_is_south, -1.0, 1.0)[:, None].astype(F32)
    g = cfg.gamma
    a = _unit(sign * g * c_N[None, :] + np.sqrt(max(1e-9, 1.0 - g * g)) * R)

    Q = M * cfg.queries_per_topic
    q_topic = np.repeat(np.arange(M), cfg.queries_per_topic)
    q = _unit(a[q_topic] + cfg.query_noise
              * rng.standard_normal((Q, d)).astype(F32))
    q_is_south = topic_is_south[q_topic]

    # ---- corpus (stochastic coverage + quality degradation) --------------
    vecs, s_info = [], []
    uncovered = 0
    for m in range(M):
        if topic_is_south[m]:
            if cfg.stochastic_coverage:
                n_info = int(rng.binomial(cfg.docs_per_north_topic, cfg.rho))
            else:
                n_info = int(round(cfg.rho * cfg.docs_per_north_topic))
            noise = cfg.doc_noise * cfg.south_doc_noise_mult
            if n_info == 0:
                uncovered += 1
        else:
            n_info = cfg.docs_per_north_topic
            noise = cfg.doc_noise
        if n_info > 0:
            v = _unit(a[m] + noise * rng.standard_normal((n_info, d)).astype(F32))
            vecs.append(v)
            s_info.append(np.full(n_info, bool(topic_is_south[m])))
    if cfg.n_distractors > 0:
        vecs.append(_unit(rng.standard_normal((cfg.n_distractors, d)).astype(F32)))
        s_info.append(np.zeros(cfg.n_distractors, dtype=bool))

    return World(
        cfg=cfg, a=a, topic_is_south=topic_is_south, c_N=c_N,
        q=q, q_topic=q_topic, q_is_south=q_is_south,
        doc_vec=np.vstack(vecs), doc_is_south_info=np.concatenate(s_info),
        frac_south_topics_uncovered=uncovered / max(1, n_south),
    )


def _retrieve_evidence(world: World, cfg: Config):
    q, D = world.q, world.doc_vec
    sims = q @ D.T
    biased = sims + cfg.beta * (D @ world.c_N)[None, :]
    k, Q = cfg.k, q.shape[0]
    rows = np.arange(Q)[:, None]

    if cfg.exposure_floor > 0:
        # deployable fair-exposure — the floor is filled by the SAME
        # biased production score, restricted to Southern-origin documents.
        south_idx = np.where(world.doc_is_south_info)[0]
        floor = int(min(cfg.exposure_floor, k, south_idx.size))
        n_fill = k - floor
        fill_scores = biased.copy()
        fill_scores[:, south_idx] = -np.inf
        fill_idx = (np.argpartition(-fill_scores, kth=n_fill - 1, axis=1)[:, :n_fill]
                    if n_fill > 0 else np.empty((Q, 0), dtype=int))
        if floor > 0:
            s_biased = biased[:, south_idx]
            top_s = np.argpartition(-s_biased, kth=floor - 1, axis=1)[:, :floor]
            idx = np.concatenate([fill_idx, south_idx[top_s]], axis=1)
        else:
            idx = fill_idx
    else:
        idx = np.argpartition(-biased, kth=k - 1, axis=1)[:, :k]

    w = biased[rows, idx] / cfg.tau
    w = np.exp(w - w.max(axis=1, keepdims=True))
    w /= w.sum(axis=1, keepdims=True)
    evid = _unit(np.einsum('qk,qkd->qd', w, D[idx]))
    return evid, float(world.doc_is_south_info[idx].sum(axis=1).mean())


def _parametric_prior(world: World, cfg: Config):
    """p = normalize((1 - s_g) a_m + s_g c_N + noise); s_S = pi, s_N = pi_north."""
    rng = np.random.default_rng(cfg.seed + 991)
    a_true = world.a[world.q_topic]
    s = np.where(world.q_is_south, cfg.pi, cfg.pi_north)[:, None].astype(F32)
    noise = cfg.prior_noise * rng.standard_normal(a_true.shape).astype(F32)
    return _unit((1.0 - s) * a_true + s * world.c_N[None, :] + noise)


def run_pipeline(world: World, cfg: Config | None = None) -> dict:
    if cfg is None:
        cfg = world.cfg
    a_true = world.a[world.q_topic]
    p = _parametric_prior(world, cfg)
    F_no = np.sum(p * a_true, axis=1)

    evid, south_ret = _retrieve_evidence(world, cfg)
    y = _unit(cfg.alpha * evid + (1.0 - cfg.alpha) * p)
    F_rag = np.sum(y * a_true, axis=1)

    s_mask, n_mask = world.q_is_south, ~world.q_is_south
    FN_rag, FS_rag = float(F_rag[n_mask].mean()), float(F_rag[s_mask].mean())
    FN_no, FS_no = float(F_no[n_mask].mean()), float(F_no[s_mask].mean())

    # per-answer chance-band statistics (95% band for a single random
    # answer: |F| <= 1.96/sqrt(d)); reported separately from the mean.
    band = 1.96 / np.sqrt(cfg.d)
    return dict(
        F_N_rag=FN_rag, F_S_rag=FS_rag, gap_rag=FN_rag - FS_rag,
        F_N_no=FN_no, F_S_no=FS_no, gap_no=FN_no - FS_no,
        rag_effect_on_gap=(FN_rag - FS_rag) - (FN_no - FS_no),
        south_uplift=FS_rag - FS_no, north_uplift=FN_rag - FN_no,
        mean_south_docs_retrieved=south_ret,
        frac_S_in_chance_band=float((np.abs(F_rag[s_mask]) <= band).mean()),
        median_abs_F_S=float(np.median(np.abs(F_rag[s_mask]))),
        frac_south_topics_uncovered=world.frac_south_topics_uncovered,
    )


def run_config(cfg: Config) -> dict:
    return run_pipeline(build_world(cfg), cfg)


KEYS = ['F_N_rag', 'F_S_rag', 'gap_rag', 'F_N_no', 'F_S_no', 'gap_no',
        'rag_effect_on_gap', 'south_uplift', 'north_uplift',
        'mean_south_docs_retrieved', 'frac_S_in_chance_band',
        'median_abs_F_S', 'frac_south_topics_uncovered']


def replicate(cfg: Config, n_seeds: int = 25, base_seed: int = 0) -> dict:
    acc = {k: [] for k in KEYS}
    for s in range(n_seeds):
        out = run_config(replace(cfg, seed=base_seed + s))
        for k in KEYS:
            acc[k].append(out[k])
    res = {'n_seeds': n_seeds}
    for k in KEYS:
        arr = np.asarray(acc[k], dtype=float)
        res[k + '_mean'] = float(arr.mean())
        res[k + '_ci'] = float(1.96 * arr.std(ddof=1) / np.sqrt(len(arr)))
        res[k + '_vals'] = arr
    return res


def t_stat_vs_zero(vals: np.ndarray):
    """Seed-level one-sample t statistic of a mean against zero."""
    se = vals.std(ddof=1) / np.sqrt(len(vals))
    return float(vals.mean() / se), se


# --------------------------------------------------------------------------- #
# Shapley attribution, parameterised by fair/biased levels
# --------------------------------------------------------------------------- #
FACTORS = ("corpus", "beta", "pi")

# The corpus factor toggles quantity AND quality together.
FAIR_LEVELS = dict(corpus=dict(rho=0.50, south_doc_noise_mult=1.0),
                   beta=dict(beta=0.0), pi=dict(pi=0.0))


def biased_levels(rho=0.05, quality_mult=2.0, beta=0.30, pi=0.60):
    return dict(corpus=dict(rho=rho, south_doc_noise_mult=quality_mult),
                beta=dict(beta=beta), pi=dict(pi=pi))


def shapley(biased: dict, seeds: int = 25, base: dict | None = None):
    base = base or {}
    subsets = [frozenset(c) for r in range(4)
               for c in itertools.combinations(FACTORS, r)]

    def value(S):
        params = dict(base)
        for f in FACTORS:
            params.update(biased[f] if f in S else FAIR_LEVELS[f])
        return float(np.mean([run_config(Config(seed=s, **params))["gap_rag"]
                              for s in range(seeds)]))

    val = {S: value(S) for S in subsets}
    phi, n = {f: 0.0 for f in FACTORS}, len(FACTORS)
    for f in FACTORS:
        for r in range(n):
            for c in itertools.combinations([g for g in FACTORS if g != f], r):
                S = frozenset(c)
                w = factorial(len(S)) * factorial(n - len(S) - 1) / factorial(n)
                phi[f] += w * (val[S | {f}] - val[S])
    total = val[frozenset(FACTORS)] - val[frozenset()]
    shares = {f: phi[f] / total for f in FACTORS}
    return phi, shares, val

"""Fast evaluation: cache world + query-doc similarity matrix per (rho, mult,
seed); beta / alpha / pi / floor only re-score, never rebuild. Numerically identical to the reference run_config (same RNG streams)."""

@lru_cache(maxsize=8)
def _cache(rho, mult, stoch, seed):
    w = build_world(Config(rho=rho, south_doc_noise_mult=mult,
                           stochastic_coverage=stoch, seed=seed))
    sims = w.q @ w.doc_vec.T
    reg = w.doc_vec @ w.c_N
    return w, sims, reg

def evaluate(cfg: Config) -> dict:
    w, sims, reg = _cache(cfg.rho, cfg.south_doc_noise_mult,
                          cfg.stochastic_coverage, cfg.seed)
    biased = sims + cfg.beta * reg[None, :]
    k, Q = cfg.k, sims.shape[0]
    rows = np.arange(Q)[:, None]
    if cfg.exposure_floor > 0:
        south_idx = np.where(w.doc_is_south_info)[0]
        floor = int(min(cfg.exposure_floor, k, south_idx.size))
        n_fill = k - floor
        fs = biased.copy(); fs[:, south_idx] = -np.inf
        fill = (np.argpartition(-fs, kth=n_fill-1, axis=1)[:, :n_fill]
                if n_fill > 0 else np.empty((Q,0), dtype=int))
        if floor > 0:
            sb = biased[:, south_idx]
            tops = np.argpartition(-sb, kth=floor-1, axis=1)[:, :floor]
            idx = np.concatenate([fill, south_idx[tops]], axis=1)
        else:
            idx = fill
    else:
        idx = np.argpartition(-biased, kth=k-1, axis=1)[:, :k]
    ww = biased[rows, idx] / cfg.tau
    ww = np.exp(ww - ww.max(axis=1, keepdims=True)); ww /= ww.sum(axis=1, keepdims=True)
    evid = _unit(np.einsum('qk,qkd->qd', ww, w.doc_vec[idx]))

    rng = np.random.default_rng(cfg.seed + 991)
    a_true = w.a[w.q_topic]
    s = np.where(w.q_is_south, cfg.pi, cfg.pi_north)[:, None].astype(np.float32)
    p = _unit((1-s)*a_true + s*w.c_N[None,:]
              + cfg.prior_noise*rng.standard_normal(a_true.shape).astype(np.float32))
    F_no = np.sum(p*a_true, axis=1)
    y = _unit(cfg.alpha*evid + (1-cfg.alpha)*p)
    F_rag = np.sum(y*a_true, axis=1)

    sm, nm = w.q_is_south, ~w.q_is_south
    FN_rag, FS_rag = float(F_rag[nm].mean()), float(F_rag[sm].mean())
    FN_no, FS_no = float(F_no[nm].mean()), float(F_no[sm].mean())
    band = 1.96/np.sqrt(cfg.d)
    return dict(F_N_rag=FN_rag, F_S_rag=FS_rag, gap_rag=FN_rag-FS_rag,
                F_N_no=FN_no, F_S_no=FS_no, gap_no=FN_no-FS_no,
                rag_effect_on_gap=(FN_rag-FS_rag)-(FN_no-FS_no),
                south_uplift=FS_rag-FS_no, north_uplift=FN_rag-FN_no,
                mean_south_docs_retrieved=float(w.doc_is_south_info[idx].sum(axis=1).mean()),
                frac_S_in_chance_band=float((np.abs(F_rag[sm])<=band).mean()),
                median_abs_F_S=float(np.median(np.abs(F_rag[sm]))),
                frac_south_topics_uncovered=w.frac_south_topics_uncovered)


# equivalence check
_c = Config(seed=3)
_a, _b = run_config(_c), evaluate(_c)
assert all(abs(_a[k]-_b[k]) < 1e-6 for k in _a), "vectorised path diverged from reference"


## 1. Main comparison

In [ ]:
def rep(params, n=None):
    """Mean and 95% CI of each outcome over independent worlds (seeds)."""
    n = (8 if FAST else 25) if n is None else n
    acc = {k: [] for k in KEYS}
    for s in range(n):
        o = evaluate(Config(seed=s, **params))
        for k in KEYS: acc[k].append(o[k])
    r = {}
    for k in KEYS:
        a = np.asarray(acc[k], float)
        r[k] = float(a.mean()); r[k+'_ci'] = float(1.96*a.std(ddof=1)/np.sqrt(n)); r[k+'_vals'] = a
    return r

FAIR = dict(rho=1.0, beta=0.0, pi=0.0, south_doc_noise_mult=1.0)
REAL = dict(rho=0.05, beta=0.30, pi=0.60, south_doc_noise_mult=2.0)
fair, real = rep(FAIR), rep(REAL)

band = 1.96/np.sqrt(Config().d)
v = real['F_S_rag_vals']; t_FS = v.mean()/(v.std(ddof=1)/np.sqrt(len(v)))
print(f"Fair world:      gap(no-RAG)={fair['gap_no']:+.3f}  gap(RAG)={fair['gap_rag']:.3f}  "
      f"RAG effect={fair['rag_effect_on_gap']:+.3f} +/- {fair['rag_effect_on_gap_ci']:.3f}")
print(f"Realistic world: F_N={real['F_N_rag']:.3f}  F_S={real['F_S_rag']:.3f}  "
      f"gap(no-RAG)={real['gap_no']:.3f}  gap(RAG)={real['gap_rag']:.3f}  "
      f"RAG effect={real['rag_effect_on_gap']:+.3f} ({real['rag_effect_on_gap']/real['gap_no']:+.1%})")
print(f"  Southern answers within the 95% chance band (|F| <= {band:.3f}): "
      f"{real['frac_S_in_chance_band']:.1%}; median |F_S| = {real['median_abs_F_S']:.3f}")
print(f"  mean F_S = {real['F_S_rag']:.3f} (seed-level t vs 0 = {t_FS:.1f})")
print(f"  Southern topics with no informative document: {real['frac_south_topics_uncovered']:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.4), sharey=True)
for ax, d_, ttl in [(axes[0], fair, "(a) Fair world"), (axes[1], real, "(b) Realistic world")]:
    FN = [d_["F_N_no"], d_["F_N_rag"]]; FS = [d_["F_S_no"], d_["F_S_rag"]]
    ax.plot([0, 1], FN, "-o", color="#1f3b73", lw=2.4, ms=7, label="North $F_N$")
    ax.plot([0, 1], FS, "-o", color="#c0392b", lw=2.4, ms=7, label="South $F_S$")
    ax.annotate("", xy=(1.07, FN[1]), xytext=(1.07, FS[1]),
                arrowprops=dict(arrowstyle="<->", color="#444", lw=1.2))
    ax.text(1.11, (FN[1]+FS[1])/2, f"gap\n{d_['gap_rag']:.2f}", va="center", ha="left", fontsize=8.5, color="#333")
    ax.axhspan(-band, band, color="#bbbbbb", alpha=0.20, zorder=0)
    ax.text(0.5, -0.05, "per-answer 95% chance band", fontsize=7, color="#777", va="center", ha="center")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["no-RAG\n(memory)", "RAG"]); ax.set_xlim(-0.18, 1.55)
    ax.set_title(ttl); ax.grid(alpha=0.2, axis="y")
axes[0].set_ylabel("answer fidelity"); axes[0].legend(loc="lower left", fontsize=8.5, frameon=False)
fig.savefig("figure1_main.png", dpi=300, bbox_inches="tight"); fig.savefig("figure1_main.pdf", bbox_inches="tight"); plt.show()

## 2. Phase maps

The RAG effect on the gap over the (retriever bias, Southern representation) plane and the
Southern uplift over the (groundedness, retriever bias) plane. The neutral contour and the
groundedness-reversal point are located per world so that confidence intervals can be
reported.

In [ ]:
G = 9 if FAST else 13
S = 3 if FAST else 6
betas  = np.linspace(0.0, 0.6, G); rhos   = np.linspace(0.02, 1.0, G)
alphas = np.linspace(0.0, 1.0, G); betas2 = np.linspace(0.0, 0.6, G)

Zeff = np.zeros((S, G, G)); Zup = np.zeros((S, G, G))
for s in range(S):
    for i, r in enumerate(rhos):
        for j, b in enumerate(betas):
            Zeff[s, i, j] = evaluate(Config(seed=s, rho=float(r), beta=float(b), pi=0.6,
                                            alpha=0.7, south_doc_noise_mult=2.0))['rag_effect_on_gap']
    _cache.cache_clear()
    for i, b in enumerate(betas2):
        for j, a in enumerate(alphas):
            Zup[s, i, j] = evaluate(Config(seed=s, rho=0.05, beta=float(b), pi=0.6,
                                           alpha=float(a), south_doc_noise_mult=2.0))['south_uplift']
    _cache.cache_clear()

def _crossings(Z):
    lo, hi = [], []
    for s in range(S):
        cr = []
        for i in range(G):
            row = Z[s, i]; jj = np.where(np.diff(np.sign(row)) > 0)[0]
            if len(jj):
                j = jj[0]; cr.append(betas[j] + (0-row[j])*(betas[j+1]-betas[j])/(row[j+1]-row[j]))
        if cr: lo.append(min(cr)); hi.append(max(cr))
    return np.array(lo), np.array(hi)

ci = lambda a: 1.96*a.std(ddof=1)/np.sqrt(len(a))
lo, hi = _crossings(Zeff)
amp = np.array([(Zeff[s] > 0).mean() for s in range(S)])
flips = []
for s in range(S):
    sl = Zup[s, :, -1] - Zup[s, :, 0]
    for i in range(G-1):
        if sl[i] > 0 and sl[i+1] <= 0:
            flips.append(betas2[i] + (0-sl[i])*(betas2[i+1]-betas2[i])/(sl[i+1]-sl[i])); break
flips = np.array(flips)
print(f"neutral contour: beta in [{lo.mean():.3f} +/- {ci(lo):.3f}, {hi.mean():.3f} +/- {ci(hi):.3f}] "
      f"across rho in [{rhos[0]:.2f}, {rhos[-1]:.2f}]")
print(f"amplifying fraction of the grid: {amp.mean():.1%} +/- {ci(amp):.1%}")
print(f"groundedness reversal at beta = {flips.mean():.3f} +/- {ci(flips):.3f}")

Ze, Zu = Zeff.mean(0), Zup.mean(0)
fig, ax = plt.subplots(1, 2, figsize=(7.8, 3.4), gridspec_kw=dict(wspace=0.45))
v = np.abs(Ze).max()
im0 = ax[0].imshow(Ze, origin="lower", aspect="auto", extent=[betas[0], betas[-1], rhos[0], rhos[-1]],
                   cmap="RdBu_r", vmin=-v, vmax=v)
cs = ax[0].contour(betas, rhos, Ze, levels=[0.0], colors="k", linewidths=1.6); ax[0].clabel(cs, fmt="neutral", fontsize=7.5)
ax[0].scatter([0.0], [1.0], marker="o", s=55, facecolor="white", edgecolor="k", zorder=5)
ax[0].annotate("fair", (0.0, 1.0), textcoords="offset points", xytext=(8, -10), fontsize=8)
ax[0].scatter([0.30], [0.05], marker="*", s=160, facecolor="yellow", edgecolor="k", zorder=5)
ax[0].annotate("operating point", (0.30, 0.05), textcoords="offset points", xytext=(6, 8), fontsize=8)
ax[0].set_xlabel(r"retriever bias  $\beta$"); ax[0].set_ylabel(r"Southern representation  $\rho$")
ax[0].set_title("(a) RAG effect on the gap"); fig.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.03)
v = np.abs(Zu).max()
im1 = ax[1].imshow(Zu, origin="lower", aspect="auto", extent=[alphas[0], alphas[-1], betas2[0], betas2[-1]],
                   cmap="RdBu_r", vmin=-v, vmax=v)
cs = ax[1].contour(alphas, betas2, Zu, levels=[0.0], colors="k", linewidths=1.6); ax[1].clabel(cs, fmt="no effect", fontsize=7.5)
ax[1].set_xlabel(r"groundedness  $\alpha$"); ax[1].set_ylabel(r"retriever bias  $\beta$")
ax[1].set_title("(b) Southern uplift"); fig.colorbar(im1, ax=ax[1], fraction=0.046, pad=0.03)
fig.savefig("figure2_phasemaps.png", dpi=300, bbox_inches="tight"); fig.savefig("figure2_phasemaps.pdf", bbox_inches="tight"); plt.show()

## 3. Attribution

Exact Shapley decomposition of the gap over the corpus, retriever, and prior factors, with
per-world confidence intervals, a document-count-only variant of the corpus factor, and a
sweep over alternative biased-level settings.

In [ ]:
FACTORS = ("corpus", "beta", "pi")
subs = [frozenset(c) for r in range(4) for c in itertools.combinations(FACTORS, r)]
FAIR_LEVELS = dict(corpus=dict(rho=1.0, south_doc_noise_mult=1.0), beta=dict(beta=0.0), pi=dict(pi=0.0))

def biased_levels(rho=0.05, quality_mult=2.0, beta=0.30, pi=0.60):
    return dict(corpus=dict(rho=rho, south_doc_noise_mult=quality_mult), beta=dict(beta=beta), pi=dict(pi=pi))

def _shares(val):
    phi = {f: 0.0 for f in FACTORS}; n = 3
    for f in FACTORS:
        for r in range(n):
            for c in itertools.combinations([g for g in FACTORS if g != f], r):
                Ss = frozenset(c); w = factorial(len(Ss))*factorial(n-len(Ss)-1)/factorial(n)
                phi[f] += w*(val[Ss | {f}] - val[Ss])
    tot = val[frozenset(FACTORS)] - val[frozenset()]
    return {f: phi[f]/tot for f in FACTORS}, tot

def shapley(biased, seeds=None, base=None, want_ci=False):
    """Exact Shapley shares of the gap. Seed-outer with a cache clear per seed."""
    seeds = (8 if FAST else 25) if seeds is None else seeds
    base = base or {}
    acc = {Ss: [] for Ss in subs}; per = []
    for s in range(seeds):
        vS = {}
        for Ss in subs:
            p = dict(base)
            for f in FACTORS: p.update(biased[f] if f in Ss else FAIR_LEVELS[f])
            g = evaluate(Config(seed=s, **p))['gap_rag']; vS[Ss] = g; acc[Ss].append(g)
        if want_ci: per.append(_shares(vS)[0])
        _cache.cache_clear()
    val = {Ss: float(np.mean(v)) for Ss, v in acc.items()}
    sh, tot = _shares(val)
    if want_ci:
        cis = {f: 1.96*np.std([p[f] for p in per], ddof=1)/np.sqrt(seeds) for f in FACTORS}
        return sh, tot, val, cis
    return sh, tot, val

sh, tot, val, cis = shapley(biased_levels(), want_ci=True)
print("Shapley shares of the gap (corpus factor toggles quantity and quality together):")
for f in ("beta", "pi", "corpus"): print(f"    {f:7s}: {sh[f]:.1%} +/- {cis[f]:.1%}")
sh0, _, _ = shapley(biased_levels(quality_mult=1.0))
print("Document-count-only corpus factor:")
for f in ("beta", "pi", "corpus"): print(f"    {f:7s}: {sh0[f]:.1%}")

sens_pts = [(b, p, r, 2.0) for b, p, r in itertools.product((0.15, 0.30, 0.45), (0.30, 0.60, 0.90), (0.02, 0.05, 0.20))] \
         + [(0.30, 0.60, 0.05, m) for m in (1.5, 3.0)]
rows = []
for b, p, r, m in sens_pts:
    s_, _, _ = shapley(biased_levels(rho=r, quality_mult=m, beta=b, pi=p), seeds=3 if FAST else 6)
    rows.append(s_)
arr = {f: np.array([x[f] for x in rows]) for f in FACTORS}
print(f"Across {len(rows)} biased-level settings:")
for f in ("beta", "pi", "corpus"):
    print(f"    {f:7s}: median {np.median(arr[f]):.1%}  range [{arr[f].min():.1%}, {arr[f].max():.1%}]")
print(f"    retriever is the largest share in {np.mean([x['beta']==max(x.values()) for x in rows]):.0%} of settings")

_Attribution figure: subset lattice and share distribution._

In [ ]:
from matplotlib.lines import Line2D
sym = {"corpus": "$C$", "beta": r"$\beta$", "pi": r"$\pi$"}
col = {"corpus": "#4C72B0", "beta": "#c0392b", "pi": "#55A868"}
order = {0: [frozenset()], 1: [frozenset({"corpus"}), frozenset({"beta"}), frozenset({"pi"})],
         2: [frozenset({"corpus","beta"}), frozenset({"corpus","pi"}), frozenset({"beta","pi"})], 3: [frozenset(FACTORS)]}
posy = {0: [0], 1: [1, 0, -1], 2: [1, 0, -1], 3: [0]}; pos = {}
for lvl in range(4):
    for Ss, y in zip(order[lvl], posy[lvl]): pos[Ss] = (lvl, y)
fig, (axL, axR) = plt.subplots(1, 2, figsize=(8.0, 3.8), gridspec_kw=dict(width_ratios=[1.8, 1.0], wspace=0.55))
for Ss in subs:
    for f in FACTORS:
        if f not in Ss:
            T = Ss | {f}; x0, y0 = pos[Ss]; x1, y1 = pos[T]; mc_ = val[T]-val[Ss]; lw = 1.0 + 8.0*max(0, mc_)/0.30
            axL.annotate("", xy=(x1, y1), xytext=(x0, y0),
                         arrowprops=dict(arrowstyle="-|>", color=col[f], lw=lw, alpha=0.75, shrinkA=15, shrinkB=15))
for Ss in subs:
    x, y = pos[Ss]; axL.scatter([x], [y], s=640, facecolor="white", edgecolor="#333", zorder=4, linewidths=1.1)
    nm = "$\\varnothing$" if len(Ss) == 0 else ",".join(sym[f] for f in FACTORS if f in Ss)
    axL.text(x, y, nm, ha="center", va="center", fontsize=9, zorder=5)
    axL.text(x, y-0.30, f"{val[Ss]:.2f}", ha="center", va="center", fontsize=7.5, color="#c0392b", zorder=5)
axL.set_xlim(-0.5, 3.5); axL.set_ylim(-1.55, 1.55); axL.set_xticks([0, 1, 2, 3])
axL.set_xticklabels(["all fair", "1 biased", "2 biased", "all biased"], fontsize=8.5); axL.set_yticks([])
axL.set_title("(a) Subset lattice of the gap", fontsize=10)
for sp in ["top", "right", "left"]: axL.spines[sp].set_visible(False)
axL.legend(handles=[Line2D([0], [0], color=col[f], lw=3, label=f"add {sym[f]}") for f in FACTORS],
           loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, fontsize=8.5, frameon=False)
labels = [("beta", "retriever bias " + sym["beta"]), ("pi", "prior skew " + sym["pi"]),
          ("corpus", "corpus " + sym["corpus"] + "\n(qty+quality)")]
yy = np.arange(len(labels))[::-1]
for (f, lab), y in zip(labels, yy):
    a = arr[f]; axR.hlines(y, a.min(), a.max(), color=col[f], lw=6, alpha=0.30); axR.scatter(np.median(a), y, color=col[f], s=45, zorder=4)
    axR.scatter(sh[f], y, color=col[f], s=110, marker="*", edgecolor="k", linewidths=0.5, zorder=5)
    axR.text(sh[f], y+0.22, f"{sh[f]:.0%}", ha="center", fontsize=8.5, color=col[f])
axR.set_yticks(yy); axR.set_yticklabels([l for _, l in labels], fontsize=8.5); axR.set_xlabel("Shapley share of the gap")
axR.set_xlim(0, 0.9); axR.set_ylim(-0.9, 2.6); axR.set_title(f"(b) Share ranges, {len(rows)} settings", fontsize=10); axR.grid(axis="x", alpha=0.25)
axR.scatter([], [], marker="*", color="#555", s=110, edgecolor="k", label="operating point"); axR.scatter([], [], marker="o", color="#555", s=45, label="median")
axR.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=8, frameon=False)
fig.savefig("figure3_attribution.png", dpi=300, bbox_inches="tight"); fig.savefig("figure3_attribution.pdf", bbox_inches="tight"); plt.show()

## 4. Mitigation

In [ ]:
seeds_m = 8 if FAST else 20
def Mv(**kw): return rep({**REAL, **kw}, n=seeds_m)
levers = {
    "baseline":            Mv(),
    "lower alpha (0.3)":    Mv(alpha=0.3),
    "exposure floor (2)":   Mv(exposure_floor=2),
    "exposure floor (3)":   Mv(exposure_floor=3),
    "augment rho (0.3)":    Mv(rho=0.30),
    "augment rho (0.5)":    Mv(rho=0.50),
    "augment rho+quality":  Mv(rho=1.0, south_doc_noise_mult=1.0),
    "debias beta (0.15)":   Mv(beta=0.15),
    "debias beta (0)":      Mv(beta=0.0),
    "combined":             Mv(beta=0.15, rho=0.20, exposure_floor=2),
}
print(f"{'lever':22s}{'F_S':>9s}{'gap':>8s}{'RAG effect':>12s}")
for k, r in levers.items():
    print(f"{k:22s}{r['F_S_rag']:7.3f}  {r['gap_rag']:7.3f}{r['rag_effect_on_gap']:+12.3f}")

gap_no = real["gap_no"]
items = {"baseline": ("baseline", "#000000", "*", 200),
 "lower alpha (0.3)": ("lower groundedness $\\alpha$=0.3", "#7f7f7f", "o", 60),
 "exposure floor (2)": ("exposure floor $f$=2", "#e08214", "s", 60),
 "exposure floor (3)": ("exposure floor $f$=3", "#b35806", "s", 60),
 "augment rho (0.3)": ("augment $\\rho$=0.3", "#4C72B0", "^", 60),
 "augment rho (0.5)": ("augment $\\rho$=0.5", "#2c5aa0", "^", 60),
 "augment rho+quality": ("augment $\\rho$=1, $\\mu_S$=1", "#123c78", "^", 70),
 "debias beta (0.15)": ("debias $\\beta$=0.15", "#c0392b", "D", 60),
 "debias beta (0)": ("debias $\\beta$=0", "#7b241c", "D", 75),
 "combined": ("combined", "#7d3c98", "P", 100)}
fig, ax = plt.subplots(figsize=(7.6, 4.0))
for k, (lab, c, mk, sz) in items.items():
    d_ = levers[k]; ax.scatter([d_["gap_rag"]], [d_["F_S_rag"]], marker=mk, s=sz, color=c, edgecolor="k", linewidths=0.6, zorder=4, label=lab)
ax.axvline(gap_no, color="#555", ls="--", lw=1.1); ax.text(gap_no+0.004, 0.002, "RAG-neutral (no-RAG gap)", fontsize=7.5, color="#555", rotation=90, va="bottom")
ax.axvspan(gap_no, 0.55, color="#c0392b", alpha=0.05)
ax.set_xlabel(r"North--South gap $\Delta_{\mathrm{RAG}}$ (lower is better)  $\rightarrow$ amplifies"); ax.set_ylabel("Southern fidelity  $F_S$")
ax.set_xlim(0.20, 0.55); ax.set_ylim(-0.01, 0.24); ax.grid(alpha=0.2)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8, frameon=False)
fig.savefig("figure4_mitigation.png", dpi=300, bbox_inches="tight"); fig.savefig("figure4_mitigation.pdf", bbox_inches="tight"); plt.show()

## 5. Robustness to Northern prior imperfection

In [ ]:
print(f"{'pi_N':>5s}{'RAG effect':>12s}{'gap(RAG)':>10s}   shares (beta/pi/corpus)")
for pn in (0.0, 0.1, 0.2, 0.3):
    r = rep({**REAL, "pi_north": pn}, n=6 if FAST else 12)
    shn, _, _ = shapley(biased_levels(), seeds=4 if FAST else 8, base=dict(pi_north=pn))
    print(f"{pn:5.1f}{r['rag_effect_on_gap']:+12.3f}{r['gap_rag']:10.3f}   "
          f"{shn['beta']:.0%} / {shn['pi']:.0%} / {shn['corpus']:.0%}")

w_ = build_world(Config(**REAL, seed=0))
top = np.partition(w_.q @ w_.doc_vec.T, -50, axis=1)[:, -50:]; sd_top = float(top.std())
print(f"\nStandard deviation of top-candidate cosine scores: {sd_top:.3f}. "
      f"The operating beta = 0.30 corresponds to {0.30/sd_top:.1f} score standard deviations; "
      f"the calibration below reports beta on the same normalised scale.")

## 6. Retriever-bias calibration on FLORES-200

For each Southern-language query the English-document advantage is measured two ways: the
same-concept advantage, and the mismatched-concept offset, which isolates the
query-independent score inflation. Requires internet; runs on Google Colab as-is.

In [ ]:
!wget -q -nc https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz -O flores200.tar.gz
!tar -xzf flores200.tar.gz
!pip -q install sentence-transformers

In [ ]:
import os, numpy as np
from sentence_transformers import SentenceTransformer

DEV_DIR = "flores200_dataset/dev"
NORTH = {"eng_Latn": "English"}
SOUTH = {"swh_Latn": "Swahili", "yor_Latn": "Yoruba", "hau_Latn": "Hausa", "amh_Ethi": "Amharic",
         "ben_Beng": "Bengali", "tgl_Latn": "Tagalog", "zul_Latn": "Zulu", "npi_Deva": "Nepali"}
N_SENT = 60

data = {}
for code_ in list(NORTH) + list(SOUTH):
    path = os.path.join(DEV_DIR, f"{code_}.dev")
    if os.path.exists(path):
        with open(path, encoding="utf-8") as fh: data[code_] = [l.strip() for l in fh][:N_SENT]
    else: print("skip", code_, "file not found")
assert "eng_Latn" in data, "English file missing; check extraction"
avail_south = [c for c in SOUTH if c in data]
nS = min(len(v) for v in data.values())
print(f"loaded {len(avail_south)} Southern languages, {nS} aligned sentences each")

In [ ]:
MODELS = {
    "MiniLM-multi": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "LaBSE": "sentence-transformers/LaBSE",
    "mE5-base": "intfloat/multilingual-e5-base",
    "MiniLM-EN": "sentence-transformers/all-MiniLM-L6-v2",
}

def calibrate(model_name):
    m = SentenceTransformer(model_name)
    pref = "query: " if "e5" in model_name else ""
    emb = {c: m.encode([pref + s for s in data[c][:nS]], normalize_embeddings=True) for c in data}
    eng = emb["eng_Latn"]; out = []
    for c in avail_south:
        q = emb[c]; others = [d_ for d_ in avail_south if d_ != c]
        m1 = np.mean((q*eng).sum(1) - np.mean([(q*emb[d_]).sum(1) for d_ in others], axis=0))
        M_eng = q @ eng.T; M_s = np.mean([q @ emb[d_].T for d_ in others], axis=0)
        off = ~np.eye(nS, dtype=bool)
        m2 = float(M_eng[off].mean() - M_s[off].mean())
        sd = float(np.concatenate([M_eng[off].ravel(), M_s[off].ravel()]).std())
        out.append((SOUTH[c], m1, m2, m2/sd))
    return out

for name, path in MODELS.items():
    try:
        rows = calibrate(path)
        m1 = np.mean([r[1] for r in rows]); m2 = np.mean([r[2] for r in rows]); m2n = np.mean([r[3] for r in rows])
        print(f"{name:14s} same-concept {m1:+.3f} | mismatched-offset {m2:+.3f} ({m2n:+.2f} sd)")
        for r in rows: print(f"   {r[0]:9s} {r[1]:+.3f} {r[2]:+.3f} ({r[3]:+.2f} sd)")
    except Exception as e:
        print(name, "failed:", str(e)[:80])